# TP — Pratique du NLP (suite) : Bag of Words et TF-IDF

**Objectifs** : transformer du texte en vecteurs numériques avec `CountVectorizer` (Bag of Words) puis `TfidfVectorizer` (TF-IDF), et comparer les deux représentations.

**Sommaire**
- Préparation
- Partie 5 — Bag of Words
- Partie 6 — TF-IDF
- Conclusion

---
## Préparation

In [ ]:
# À décommenter si scikit-learn ou pandas ne sont pas installés
# !pip install scikit-learn pandas

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

---
## Partie 5 — Représentation Bag of Words

**Objectif** : convertir plusieurs phrases en vecteurs numériques avec `CountVectorizer`.

### 5.1 Corpus de 3 phrases
*Trois thèmes différents : football, cinéma, tennis. Vous pouvez remplacer ces phrases par les vôtres.*

In [ ]:
phrases = [
    "The football team won the match and the fans celebrated the victory",
    "The new movie won an award and the audience loved the story",
    "The tennis player trained hard before the final match",
]

for i, p in enumerate(phrases, start=1):
    print(f"P{i} : {p}")

### 5.2 Transformation avec CountVectorizer

In [ ]:
count_vec = CountVectorizer()
X_bow = count_vec.fit_transform(phrases)   # apprend le vocabulaire puis transforme

print("Forme de la matrice (phrases x mots) :", X_bow.shape)

### 5.3 Liste des mots extraits (vocabulaire)

In [ ]:
mots = count_vec.get_feature_names_out()
print(len(mots), "mots :")
print(list(mots))

### 5.4 Matrice Bag of Words

In [ ]:
# Version brute (tableau NumPy)
print(X_bow.toarray())

In [ ]:
# Version lisible : une ligne par phrase, une colonne par mot
df_bow = pd.DataFrame(X_bow.toarray(), columns=mots, index=["P1", "P2", "P3"])
df_bow

### 5.5 Questions

**Que représente chaque colonne de la matrice ?**
Chaque colonne correspond à **un mot unique du vocabulaire** (classés par ordre alphabétique). Chaque ligne correspond à une phrase, et la valeur est le **nombre d'occurrences** du mot dans cette phrase. Par exemple, la colonne `the` vaut 4 pour P1, 3 pour P2 et 2 pour P3.

**Quelles sont les limites du modèle Bag of Words ?**
- Il ignore l'**ordre des mots** et le contexte.
- Il ne capte pas le **sens** : *movie* et *film* sont deux colonnes sans lien.
- La matrice est **grande et creuse** (beaucoup de zéros), et grossit avec le vocabulaire.
- Tous les mots comptent de la même façon : les mots très fréquents comme `the` dominent les vecteurs.
- Les mots absents du vocabulaire d'apprentissage sont **ignorés** sur de nouveaux textes.

**Que perd-on en supprimant l'ordre des mots ?**
On perd la **syntaxe et le sens** qui dépendent de l'ordre. Par exemple « the dog bites the man » et « the man bites the dog » donnent exactement le même vecteur. On perd aussi les négations (« not good ») et les expressions (« New York »). Les **n-grammes** (`ngram_range=(1, 2)`) récupèrent une partie de cette information, comme le montre la cellule suivante.

In [ ]:
# Illustration : deux phrases d'ordre différent, même vecteur
a = "the dog bites the man"
b = "the man bites the dog"
vec = CountVectorizer().fit([a, b])
print("Vecteurs identiques :", (vec.transform([a]).toarray() == vec.transform([b]).toarray()).all())

# Les bigrammes retrouvent une partie de l'ordre
vec2 = CountVectorizer(ngram_range=(2, 2)).fit([a, b])
print("Bigrammes identiques :", (vec2.transform([a]).toarray() == vec2.transform([b]).toarray()).all())
print(list(vec2.get_feature_names_out()))

---
## Partie 6 — Représentation TF-IDF

**Objectif** : utiliser la pondération TF-IDF pour valoriser les mots importants.

**Rappel** : `TF-IDF = TF × IDF`
- **TF** (Term Frequency) : fréquence du mot dans le document.
- **IDF** (Inverse Document Frequency) : plus un mot apparaît dans beaucoup de documents, plus son IDF est faible. Avec scikit-learn : `idf = ln((1 + n) / (1 + df)) + 1`, où `n` est le nombre de documents et `df` le nombre de documents contenant le mot.
- Les vecteurs obtenus sont ensuite normalisés (norme L2).

### 6.1 Transformation avec TfidfVectorizer
*On reprend exactement les mêmes phrases.*

In [ ]:
tfidf_vec = TfidfVectorizer()
X_tfidf = tfidf_vec.fit_transform(phrases)

print("Forme de la matrice :", X_tfidf.shape)

### 6.2 Matrice TF-IDF

In [ ]:
df_tfidf = pd.DataFrame(
    X_tfidf.toarray(),
    columns=tfidf_vec.get_feature_names_out(),
    index=["P1", "P2", "P3"],
).round(3)
df_tfidf

### 6.3 Comparaison des poids : BoW vs TF-IDF (phrase P1)

In [ ]:
comparaison = pd.DataFrame({
    "BoW (comptage)": df_bow.loc["P1"],
    "TF-IDF": df_tfidf.loc["P1"],
})
# on garde uniquement les mots présents dans P1
comparaison = comparaison[comparaison["BoW (comptage)"] > 0]
comparaison.sort_values("TF-IDF", ascending=False)

### 6.4 Poids IDF de chaque mot
*Plus l'IDF est bas, moins le mot est discriminant.*

In [ ]:
idf = pd.Series(tfidf_vec.idf_, index=tfidf_vec.get_feature_names_out()).round(3)
idf.sort_values()

### 6.5 Mots les plus importants par phrase

In [ ]:
for phrase in df_tfidf.index:
    top = df_tfidf.loc[phrase].sort_values(ascending=False).head(3)
    print(phrase, "->", ", ".join(f"{m} ({p})" for m, p in top.items()))

### 6.6 Effet des stopwords sur TF-IDF

Dans la matrice précédente, `the` garde le poids le plus fort dans chaque phrase, malgré son IDF minimal (voir 6.4). La raison : il apparaît 2 à 4 fois par phrase, donc son TF est élevé, et l'IDF de scikit-learn ne descend jamais sous 1. En retirant les stopwords, les mots vraiment spécifiques ressortent.

In [ ]:
tfidf_sw = TfidfVectorizer(stop_words="english")
X_sw = tfidf_sw.fit_transform(phrases)

df_sw = pd.DataFrame(
    X_sw.toarray(),
    columns=tfidf_sw.get_feature_names_out(),
    index=["P1", "P2", "P3"],
).round(3)

for phrase in df_sw.index:
    top = df_sw.loc[phrase].sort_values(ascending=False).head(3)
    print(phrase, "->", ", ".join(f"{m} ({p})" for m, p in top.items()))

### 6.7 Questions

**Quelle est la différence principale entre BoW et TF-IDF ?**
Le Bag of Words **compte** simplement les occurrences de chaque mot. TF-IDF **pondère** ce comptage par la rareté du mot dans le corpus : un mot présent dans tous les documents est pénalisé, un mot présent dans peu de documents est valorisé. Ici, `football` (1 document sur 3) a un IDF de 1,693, contre 1,0 pour `the` (3 documents sur 3).

**Pourquoi les mots très fréquents ont-ils souvent un poids faible ?**
Parce que l'IDF diminue quand le mot apparaît dans beaucoup de documents : un mot présent partout ne permet pas de distinguer un document d'un autre. Attention à la nuance observée en 6.5 : le poids final dépend aussi du TF. Un mot fréquent *dans tout le corpus* mais aussi très répété dans un document (comme `the`) peut garder un poids élevé. C'est pourquoi on combine TF-IDF avec la suppression des stopwords (6.6).

**Exemple concret où TF-IDF est plus utile que BoW :**
Dans un **moteur de recherche** ou un classement d'articles de presse, avec la requête « football », le BoW favorise les documents qui contiennent le plus de mots communs ou répétés. TF-IDF met en avant les documents où « football » est un mot distinctif et peu courant dans le reste du corpus. C'est aussi utile pour extraire les **mots-clés** d'un document ou classer des textes par thème.

---
## Conclusion

| | Bag of Words | TF-IDF |
|---|---|---|
| Valeur d'une case | Nombre d'occurrences | Poids = fréquence × rareté |
| Mots communs | Dominent le vecteur | Pénalisés (surtout avec les stopwords retirés) |
| Mots distinctifs | Même poids que les autres | Valorisés |
| Ordre des mots | Perdu | Perdu (sauf avec les n-grammes) |
| Classe scikit-learn | `CountVectorizer` | `TfidfVectorizer` |

TF-IDF est en général un meilleur point de départ pour la recherche d'information et la classification de textes. Les deux méthodes restent limitées car elles ne capturent ni l'ordre ni le sens : les *embeddings* (Word2Vec, BERT) répondent à ce problème.